In [41]:
from core.dataset import Dataset
from core.configs import PreprocessingConfig
import core.experiment
import numpy as np
from scripts.analyze_mlflow_runs import (
    analyze_unified_mlruns,
    aggregate_metrics,
    print_summary,
    plot_metrics
)
from helpers import utils
from scripts import diagnose_fold_differences as fold_dx
import re
from collections import defaultdict
from reload_recursive import reload_recursive
from pathlib import Path

In [42]:
reload_recursive(core.experiment)
from core.experiment import Experiment
reload_recursive(utils)


In [3]:
dataset_name = "roi_train2"
ds = Dataset(dataset_name)
runs = {}
for i in [1,2,3,4,5,6,7,8,9]:
    exp = Experiment.from_run_dir(f"stage1_crop_lr_sweep/run{i}", ds)
    # exp.predict()
    runs[i] = exp

In [7]:
compiled_metrics = defaultdict(list)
for i, exp in runs.items():
    compiled_metrics['run'].append(i)
    compiled_metrics['lr'].append(exp.training_config.learning_rate)
    crop_ratios = exp.training_config.crop_ratios
    if crop_ratios is not None:
        compiled_metrics['crop_ratios'].append(",".join([str(x) for x in crop_ratios]))
    else:
        compiled_metrics['crop_ratios'].append("null")
    fold_data = analyze_unified_mlruns(exp.run_dir)
    aggregated = aggregate_metrics(fold_data)
    by_prefix = defaultdict(dict)
    for metric_name in aggregated:
        if "/" in metric_name:
            prefix = metric_name.split("/")[0]
            by_prefix[prefix][metric_name] = aggregated[metric_name]
        else:
            by_prefix["other"][metric_name] = aggregated[metric_name]

    for prefix in sorted(by_prefix.keys()):
        for metric_name in sorted(by_prefix[prefix].keys()):
            metric_data = aggregated[metric_name]

            if "stats" not in metric_data:
                continue

            stats = metric_data["stats"]
            for stat in ['mean', 'std', 'min', 'max']:
                compiled_metrics[f"{metric_name}_{stat}"].append(stats[stat])
            
            fold_values = [(fold_num, metric_data[fold_num])
                          for fold_num in sorted(fold_data.keys())
                          if fold_num in metric_data]
            if "val_class" not in metric_name:
                continue
            for fold_num, values in fold_values:
                compiled_metrics[f'fold{fold_num}-{metric_name}_final'].append(values[-1])
                compiled_metrics[f'fold{fold_num}-{metric_name}_max'].append(np.max(values))
            

In [ ]:
import pandas as pd
metrics = pd.DataFrame(compiled_metrics).set_index("run")
col_remap = {}
for col in metrics.columns:
    # lesion accs
    new_col = re.sub(r"^(.*)(val_)(class/acc_0)(_.+)",r"\1lesion/acc\4", col)
    col_remap[col] = new_col
    # rim accs
    col_remap[col] = re.sub(r"^(.*)(val_)(class/acc_1)(_.+)",r"\1rim/acc\4", new_col)

metrics = metrics.rename(columns=col_remap)
col_order_cats = defaultdict(list)
for col in metrics.columns:
    if "acc_min" in col or "loss_max" in col:
        continue
    if "rim" in col:
        col_order_cats['rim'].append(col)
    elif "lesion" in col:
        col_order_cats['lesion'].append(col)
    elif "train" in col:
        col_order_cats['loss'].append(col)
    elif "val" in col:
        col_order_cats['val'].append(col)
    else:
        col_order_cats['run_info'].append(col)

column_order = []
for cat in ["run_info", "rim", "lesion", "loss", "val"]:
    column_order.extend(col_order_cats[cat])
metrics = metrics[column_order]
metrics.to_csv("gridsearch_metrics.csv")
metrics

,lr,crop_ratios,rim/acc_mean,rim/acc_std,rim/acc_max,fold0-rim/acc_final,fold0-rim/acc_max,fold1-rim/acc_final,fold1-rim/acc_max,fold2-rim/acc_final,...,fold3-lesion/acc_final,fold3-lesion/acc_max,fold4-lesion/acc_final,fold4-lesion/acc_max,train/loss_mean,train/loss_std,train/loss_min,val/acc_mean,val/acc_std,val/acc_max
run,,,,,,,,,,,,,,,,,,,,,
1,0.0001,null,0.308173,0.124696,0.570181,0.335917,0.465463,0.464623,0.570181,0.203483,...,0.677204,0.683921,0.655559,0.667323,0.613500,0.060620,0.562069,0.478029,0.078950,0.631602
2,0.0002,null,0.315485,0.141505,0.596328,0.349683,0.513900,0.534867,0.596328,0.121242,...,0.678887,0.687646,0.656058,0.666753,0.615031,0.058647,0.560434,0.482145,0.085820,0.647753
3,0.0005,null,0.281576,0.130209,0.552221,0.293713,0.481362,0.494162,0.552221,0.162796,...,0.689028,0.691284,0.652374,0.659792,0.623453,0.060448,0.562251,0.461891,0.081831,0.619870
4,0.0001,"1,1,4",0.323610,0.131309,0.576896,0.309684,0.505218,0.490422,0.576896,0.145543,...,0.678992,0.688557,0.666631,0.672812,0.612527,0.060171,0.561474,0.485819,0.079501,0.625283
5,0.0002,"1,1,4",0.322764,0.135008,0.577337,0.325839,0.497418,0.536958,0.577337,0.155911,...,0.685979,0.691363,0.661829,0.672573,0.614474,0.059042,0.560221,0.484440,0.082516,0.637784
6,0.0005,"1,1,4",0.299643,0.139100,0.572368,0.382836,0.463863,0.529972,0.572368,0.145293,...,0.682172,0.689400,0.659236,0.664936,0.622458,0.060591,0.562636,0.470799,0.086113,0.632828
7,0.0001,"1,1,8",0.326156,0.122305,0.584541,0.337510,0.476146,0.506607,0.584541,0.231993,...,0.685021,0.689092,0.660642,0.673102,0.612058,0.060109,0.560673,0.487425,0.078383,0.640764
8,0.0002,"1,1,8",0.323284,0.136350,0.588553,0.328112,0.475149,0.529809,0.588553,0.183315,...,0.685212,0.690571,0.654317,0.666184,0.613792,0.058963,0.560073,0.484919,0.082765,0.640588
9,0.0005,"1,1,8",0.297522,0.141540,0.573966,0.301211,0.473860,0.530390,0.573966,0.130627,...,0.689682,0.699758,0.660242,0.666404,0.621659,0.060367,0.560307,0.470535,0.085850,0.632549


In [52]:
metrics2 = metrics.copy()
metrics2.insert(2, "rim_dice_mean (PRL test cases)", None)
metrics2.insert(3, "rim_dice_std (PRL test cases)", None)
metrics2.insert(4, "lesion_dice_mean (PRL test cases)", None)
metrics2.insert(5, "lesion_dice_std (PRL test cases)", None)

for i, exp in runs.items():
    testing_cases = [item for item in exp.cases if item['split'] == "testing" and item["case_type"] == "PRL"]
    rim_dice = []
    lesion_dice = []
    for item in testing_cases:
        rim_dice.append(utils.dice_score(item['label'], item['inference'], seg1_val=2, seg2_val=2))
        lesion_dice.append(utils.dice_score(item['label'], item['inference'], seg1_val=1, seg2_val=1))
    metrics2.loc[i, "rim_dice_mean (PRL test cases)"] = np.mean(rim_dice)
    metrics2.loc[i, "rim_dice_std (PRL test cases)"] = np.std(rim_dice)
    metrics2.loc[i, "lesion_dice_mean (PRL test cases)"] = np.mean(lesion_dice)
    metrics2.loc[i, "lesion_dice_std (PRL test cases)"] = np.std(lesion_dice)

In [54]:
metrics2.to_csv("gridsearch_metrics_w_dice.csv")

In [ ]:
run_dir = runs[1].run_dir
datastats = fold_dx.load_datastats(run_dir)
rows = []
for item in runs[1].cases:
    if item['split'] == "testing":
        continue
    rows.append({
        k: item[k] for k in ["subid", "lesion_index", "split", "case_type"]   
    })
metadata_df = pd.DataFrame(rows)
df = metadata_df.merge(datastats, on=["subid", "lesion_index"], how="left")

Loading /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/datastats_by_case.yaml ...
  Loaded 794 cases from datastats


In [32]:
all_numerical_cols = df.select_dtypes(include='number').columns
ignore_numerical_cols = ["subid", "lesion_index"]
cols_to_mean = all_numerical_cols[~all_numerical_cols.isin(ignore_numerical_cols)]
means_by_fold = df.groupby("split")[cols_to_mean].mean()

In [55]:
means_by_fold.to_csv("fold_image_stats.csv")

In [56]:
col_remap = {
        'val_class/acc_0_mean': 'val_lesion/acc_mean',
       'val_class/acc_0_std': 'val_lesion/acc_std', 
       'val_class/acc_0_min': 'val_lesion/acc_min', 
       'val_class/acc_0_max': 'val_lesion/acc_max',
       'val_class/acc_1_mean': 'val_rim/acc_mean', 
       'val_class/acc_1_std': 'val_rim/acc_std', 
       'val_class/acc_1_min': 'val_rim/acc_min',
       'val_class/acc_1_max': 'val_rim/acc_max'
}
col_order = ['lr', 'crop_ratios', 
             'val_rim/acc_mean', 'val_rim/acc_max', 'val_rim/acc_std',
             'val_lesion/acc_mean', 'val_lesion/acc_max', 'val_lesion/acc_std', 
            'train/loss_mean', 'train/loss_min', 'train/loss_std',
            'val/acc_mean', 'val/acc_max', 'val/acc_std',  
       
       ]
metrics = metrics.rename(columns=col_remap)[col_order]


In [62]:
round_columns = metrics.columns[~metrics.columns.isin(["lr", "run", "crop_ratios"])]
metrics[round_columns] = metrics[round_columns].round(3)

In [ ]:
col_remap = {
    'train/loss_mean', 'train/loss_std',
       'train/loss_min', 'train/loss_max', 'val/acc_mean', 'val/acc_std',
       'val/acc_min', 'val/acc_max', 'val_lesion/acc_mean',
       'val_lesion/acc_std', 'val_lesion/acc_min', 'val_lesion/acc_max',
       'val_rim/acc_mean', 'val_rim/acc_std', 'val_rim/acc_min',
       'val_rim/acc_max'
}

Index(['lr', 'crop_ratios', 'train/loss_mean', 'train/loss_std',
       'train/loss_min', 'train/loss_max', 'val/acc_mean', 'val/acc_std',
       'val/acc_min', 'val/acc_max', 'val_class/acc_0_mean',
       'val_class/acc_0_std', 'val_class/acc_0_min', 'val_class/acc_0_max',
       'val_class/acc_1_mean', 'val_class/acc_1_std', 'val_class/acc_1_min',
       'val_class/acc_1_max'],
      dtype='object')

In [40]:
performance_metrics = runs[1].evaluate()

/media/smbshare/srs-9/prl_project/data/sub2131-20190117/26/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub2131-20190117/26/flair.phase_xy20_z2_ensemble.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1010-20180208/27/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub1010-20180208/27/flair.phase_xy20_z2_ensemble.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1293-20161129/70/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub1293-20161129/70/flair.phase_xy20_z2_ensemble.nii.gz
194 valid cases of 197
/media/smbshare/srs-9/prl_project/data/sub1050-20170624/86/lesion_xy20_z2.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/fold_predictions/fold0/sub1050-20170624/86/flair.phase_xy20_z2.nii.gz
/media/smbshare/srs-9/prl_project/data/sub1050-

In [42]:
runs[1].cases[0]

{'subid': 1293,
 'lesion_index': 1,
 'image': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1293-20161129/1/flair.phase_xy20_z2.nii.gz'),
 'label': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1293-20161129/1/prl_label_SRS_CH_xy20_z2.nii.gz'),
 'split': 'testing',
 'inference': PosixPath('/media/smbshare/srs-9/prl_project/training/roi_train2/stage1_crop_lr_sweep/run1/ensemble_output/sub1293-20161129/1/flair.phase_xy20_z2_ensemble.nii.gz')}

In [41]:
performance_metrics

,subid,lesion_index,split,case_type,lesion_dice,prl_dice,tp,fp,tn,fn,sensitivity,specificity,fpr,fnr,precision,npv,accuracy,f1,Notes
0,1293,1,testing,None,0.888445,0.328682,106,152,7347,182,0.368056,0.979731,0.020269,0.631944,0.410853,0.975827,0.957108,0.388278,
1,1074,3,testing,None,0.678937,0.509165,125,76,166,34,0.786164,0.685950,0.314050,0.213836,0.621891,0.830000,0.725686,0.694444,
2,1080,3,testing,None,0.706790,0.528302,140,116,229,20,0.875000,0.663768,0.336232,0.125000,0.546875,0.919679,0.730693,0.673077,
3,2131,1,testing,None,0.866130,0.695710,519,198,1184,62,0.893287,0.856729,0.143271,0.106713,0.723849,0.950241,0.867550,0.799692,
4,1011,7,testing,None,0.765677,0.570922,161,100,232,9,0.947059,0.698795,0.301205,0.052941,0.616858,0.962656,0.782869,0.747100,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,1078,12,validation fold4,None,0.761905,0.000000,0,1,168,0,NaN,0.994083,0.005917,NaN,0.000000,1.000000,0.994083,NaN,
987,1050,82,validation fold4,None,0.285714,NaN,0,0,1,0,NaN,1.000000,0.000000,NaN,NaN,1.000000,1.000000,NaN,
988,1293,67,validation fold4,None,0.615385,NaN,0,0,4,0,NaN,1.000000,0.000000,NaN,NaN,1.000000,1.000000,NaN,
989,1125,96,validation fold4,None,0.602410,NaN,0,0,150,0,NaN,1.000000,0.000000,NaN,NaN,1.000000,1.000000,NaN,
